## PLUTO – Main Simulation

In this notebook, we visualize the progress in our main simulation runs.

### 0. Definitions

#### 0.1. Preamble

In [1]:
%%capture

%config InlineBackend.figure_formats = ['retina']

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

import pyPLUTO as pp

import io
import base64

from PIL import Image
from IPython.display import Image as HTML, display
import IPython.display as ipd

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import Normalize, LogNorm
from matplotlib.ticker import ScalarFormatter, MultipleLocator
from matplotlib.patches import Wedge

from skimage.filters import threshold_otsu
from scipy.ndimage import uniform_filter

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=["steelblue", "olivedrab", "goldenrod", "firebrick", "rebeccapurple"]) 
plt.rcParams['figure.figsize'] = [8,5]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["ytick.minor.visible"] = True

plt.plot()
plt.close()

#### 0.2. Helpers

In [2]:
def compute_flux_function(D):
    Br = D.Bx1
    theta = D.x2
    r = D.x1
    integrand = Br * (r[:, None]**2) * np.sin(theta)[None, :]
    Psi = np.zeros_like(integrand)
    dtheta = np.diff(theta)
    Psi[:, 1:] = np.cumsum(
        0.5 * (integrand[:, 1:] + integrand[:, :-1]) * dtheta[None, :],
        axis=1
    )
    return Psi



def animate_density(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    
    rho_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
    
    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])
    
    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20
    
    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
    
        im = ax.pcolormesh(
            X, Z, rho_frames[i], 
            cmap='magma', 
            norm=LogNorm(vmin=vmin, vmax=vmax), 
            shading='auto'
        )
    
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
    
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')
    
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')
    
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/density.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_velocity(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist
    
    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    
    rho_frames = {}
    vx_frames = {}
    vz_frames = {}
    vr_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
    
        vr = Di.vx1
        vth = Di.vx2
        vx = vr * np.sin(Theta) + vth * np.cos(Theta)
        vz = vr * np.cos(Theta) - vth * np.sin(Theta)
    
        vx_frames[i] = np.where(mask, vx, np.nan)
        vz_frames[i] = np.where(mask, vz, np.nan)
        vr_frames[i] = np.where(mask, vr, np.nan)
    
    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])
    
    stride1, stride2 = 1, 1
    
    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        im = ax.pcolormesh(
            X, Z, rho_frames[i],
            cmap='gray',
            norm=LogNorm(vmin=vmin, vmax=vmax),
            shading='auto'
        )
        Xq = X[::stride1, ::stride2]
        Zq = Z[::stride1, ::stride2]
        Uq = vx_frames[i][::stride1, ::stride2]
        Wq = vz_frames[i][::stride1, ::stride2]
        Vrq = vr_frames[i][::stride1, ::stride2]
    
        colors = np.where(Vrq >= 0, 'r', 'b').ravel()
    
        ax.quiver(
            Xq, Zq, Uq, Wq,
            color=colors,
            scale_units='xy',
            angles='xy',
        )
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/velocity.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_field(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)

    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)

    rho_frames = {}
    Psi_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
        Psi_frames[i] = compute_flux_function(Di)

    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])

    global_absmax = max(np.nanmax(np.abs(p)) for p in Psi_frames.values())
    levels_pos = np.linspace(0, global_absmax, 100)
    levels_neg = -levels_pos[::-1]

    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')

        im = ax.pcolormesh(
            X, Z, rho_frames[i],
            cmap='gray',
            norm=LogNorm(vmin=vmin, vmax=vmax),
            shading='auto'
        )

        ax.contour(X, Z, Psi_frames[i], levels=levels_pos, colors='m', linestyle='-', linewidths=0.4)
        ax.contour(X, Z, Psi_frames[i], levels=levels_neg, colors='m', linestyle='-', linewidths=0.4)

        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    gif_path = f'{path}/Storage/field.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )

    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_tracer(path, mode='truth', compare=False, panel=False):

    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist
    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)

    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)

    tracer_frames = {}
    disagreement_frames = {}

    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        if panel:
            abs_diff = np.abs(Di.diskfrac - Di.tr1)
            n_theta = abs_diff.shape[1]
            disagreement_frames[i] = np.sum(abs_diff, axis=1) / n_theta
        else:
            if compare:
                tracer = Di.diskfrac - Di.tr1
            else:
                tracer = Di.diskfrac if mode == 'recover' else Di.tr1
            tracer_frames[i] = np.where(mask, tracer, np.nan)

    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    def _save_gif(frames, gif_name, width=750):
        gif_path = f'{path}/Storage/{gif_name}'
        frames[0].save(
            gif_path,
            format='GIF',
            save_all=True,
            append_images=frames[1:],
            duration=150,
            loop=0
        )
        ipd.display(ipd.HTML(f'<img src="{gif_path}" width="{width}">'))

    if panel:
        x1 = D_last.x1
        n1 = D_last.rho.shape[0]
        comp_frames = []
        for i in outlist:
            disagreement_pct = disagreement_frames[i] * 100.0
            total_disagreement_pct = (np.sum(disagreement_frames[i]) / n1) * 100.0
            total_agreement_pct = 100.0 - total_disagreement_pct

            fig, ax = plt.subplots(figsize=[7, 4])
            ax.fill_between(x1, disagreement_pct, -5, color='m', alpha=1/4, linewidth=0)
            ax.set_xscale('log')
            ax.set_xticks([2, 4, 6, 8, 10, 20, 30])
            ax.xaxis.set_major_formatter(ScalarFormatter())
            ax.minorticks_off()
            ax.xaxis.set_minor_formatter(plt.NullFormatter())
            ax.set_yticks([0, 5, 10, 15, 20])
            ax.set_xlabel('R')
            ax.set_ylabel('|A – B| %')
            ax.set_ylim(-5, 20)

            if 'STAR' in path:
                ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
            elif 'BH' in path:
                ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
            ax.legend(loc='upper right')

            ax.text(0.5, 0.05, f'1 – |A – B| = {total_agreement_pct:.1f} %',
                     transform=ax.transAxes, ha='center', va='bottom', fontsize=10)

            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
            plt.close(fig)
            buf.seek(0)
            img = Image.open(buf)
            img.load()
            comp_frames.append(img.convert('RGB'))

        _save_gif(comp_frames, 'tracer_comparison.gif', width=650)
    else:
        def _render_field_gif(field_frames, gif_name, vmin, vmax, is_diff=False):
            frames = []
            for i in outlist:
                fig, ax = plt.subplots(figsize=[6, 8])
                ax.set_facecolor('k')

                im = ax.pcolormesh(
                    X, Z, field_frames[i],
                    cmap='coolwarm',
                    vmin=vmin, vmax=vmax,
                    shading='auto'
                )

                ax.set_xlabel('R')
                ax.set_ylabel('z')
                ax.set_aspect('equal')
                ax.set_xlim(0, 20)
                ax.set_ylim(0, 10)

                if 'STAR' in path:
                    ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
                    ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
                elif 'BH' in path:
                    ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
                    ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
                ax.legend(loc='upper right', labelcolor='w')

                divider = make_axes_locatable(ax)
                if is_diff:
                    cax = divider.append_axes("right", size="5%", pad=0.1)
                else:
                    cax = divider.append_axes("right", size="5%", pad=0.4)
                cbar = fig.colorbar(im, cax=cax)

                if is_diff:
                    cbar.set_label('A – B')
                else:
                    cbar.set_ticks([])
                    cax.text(0.5, 1.02, 'disk', transform=cax.transAxes,
                              ha='center', va='bottom', color='k', fontsize=10)
                    cax.text(0.5, -0.02, 'corona', transform=cax.transAxes,
                              ha='center', va='top', color='k', fontsize=10)

                buf = io.BytesIO()
                fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
                plt.close(fig)
                buf.seek(0)
                img = Image.open(buf)
                img.load()
                frames.append(img.convert('RGB'))

            _save_gif(frames, gif_name)

        if compare:
            gif_filename = 'tracer_diff.gif'
            _render_field_gif(tracer_frames, gif_filename, vmin=-1, vmax=1, is_diff=True)
        else:
            gif_filename = f'tracer_{mode}.gif'
            _render_field_gif(tracer_frames, gif_filename, vmin=0, vmax=1, is_diff=False)



def animate_panel(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    def profiles(D):
        rho = D.rho
        prs = D.prs
        vr = D.vx1
        vphi = D.vx3
        tr = D.tr1
        theta = D.x2
        r = D.x1

        sin_th = np.sin(theta)[None, :]
        l_specific = r[:, None] * np.sin(theta)[None, :] * vphi

        def weighted_avg(field, weight):
            num = np.trapezoid(field * weight * sin_th, theta, axis=1)
            den = np.trapezoid(weight * sin_th, theta, axis=1)
            return num / np.where(den == 0, np.nan, den)

        rho_disk = weighted_avg(rho, tr)
        rho_corona = weighted_avg(rho, 1 - tr)
        prs_disk = weighted_avg(prs, tr)
        prs_corona = weighted_avg(prs, 1 - tr)

        base = rho * vr * sin_th
        base_l = rho * vr * l_specific * sin_th
        Mdot_disk = 2 * np.pi * r**2 * np.trapezoid(base * tr, theta, axis=1)
        Mdot_corona = 2 * np.pi * r**2 * np.trapezoid(base * (1 - tr), theta, axis=1)
        Ldot_disk = 2 * np.pi * r**2 * np.trapezoid(base_l * tr, theta, axis=1)
        Ldot_corona = 2 * np.pi * r**2 * np.trapezoid(base_l * (1 - tr), theta, axis=1)

        return (r, rho_disk, rho_corona, prs_disk, prs_corona,
                Mdot_disk, Mdot_corona, Ldot_disk, Ldot_corona)

    r_ref = None
    rho_disk_frames, rho_corona_frames = {}, {}
    prs_disk_frames, prs_corona_frames = {}, {}
    Mdot_disk_frames, Mdot_corona_frames = {}, {}
    Ldot_disk_frames, Ldot_corona_frames = {}, {}

    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        r, rd, rc, pd, pc, md, mc, ld, lc = profiles(Di)
        r_ref = r
        rho_disk_frames[i] = rd
        rho_corona_frames[i] = rc
        prs_disk_frames[i] = pd
        prs_corona_frames[i] = pc
        Mdot_disk_frames[i] = md
        Mdot_corona_frames[i] = mc
        Ldot_disk_frames[i] = ld
        Ldot_corona_frames[i] = lc

    def shared_ylim(*frame_dicts, log=False):
        all_vals = np.concatenate([v[~np.isnan(v)] for d in frame_dicts for v in d.values()])
        if log:
            all_vals = all_vals[all_vals > 0]
            ymin, ymax = all_vals.min(), all_vals.max()
            pad = (ymax / ymin) ** 0.05
            return ymin / pad, ymax * pad
        ymin, ymax = all_vals.min(), all_vals.max()
        pad = 0.05 * (ymax - ymin)
        return ymin - pad, ymax + pad

    rho_ymin, rho_ymax = shared_ylim(rho_disk_frames, rho_corona_frames, log=True)
    prs_ymin, prs_ymax = shared_ylim(prs_disk_frames, prs_corona_frames, log=True)
    mdot_ymin, mdot_ymax = shared_ylim(Mdot_disk_frames, Mdot_corona_frames)
    ldot_ymin, ldot_ymax = shared_ylim(Ldot_disk_frames, Ldot_corona_frames)

    outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10

    frames = []
    for i in outlist:
        fig, axs = plt.subplots(4, 2, figsize=[7.5, 10], sharex=True)

        for col, frame_dict in [(0, rho_disk_frames), (1, rho_corona_frames)]:
            ax = axs[0, col]
            y = frame_dict[i]
            ax.fill_between(r_ref, y, rho_ymin, color='m', alpha=1/4, linewidth=0)
            ax.set_yscale('log')
            ax.set_ylim(rho_ymin, rho_ymax)
            if col == 0:
                ax.set_ylabel(r'$\langle\rho\rangle$')

        for col, frame_dict in [(0, prs_disk_frames), (1, prs_corona_frames)]:
            ax = axs[1, col]
            y = frame_dict[i]
            ax.fill_between(r_ref, y, prs_ymin, color='m', alpha=1/4, linewidth=0)
            ax.set_yscale('log')
            ax.set_ylim(prs_ymin, prs_ymax)
            if col == 0:
                ax.set_ylabel(r'$\langle P\rangle$')

        panel_spec = [
            (2, 0, Mdot_disk_frames,   r'$\dot{M}$',    mdot_ymin, mdot_ymax),
            (2, 1, Mdot_corona_frames, None,            mdot_ymin, mdot_ymax),
            (3, 0, Ldot_disk_frames,   r'$\dot{L}$',    ldot_ymin, ldot_ymax),
            (3, 1, Ldot_corona_frames, None,            ldot_ymin, ldot_ymax),
        ]
        for row, col, frame_dict, ylabel, ymin, ymax in panel_spec:
            ax = axs[row, col]
            y = frame_dict[i]
            ax.axhline(0, color='k', lw=0.8, ls='-')
            ax.fill_between(r_ref, y, 0, where=(y < 0), color='b', alpha=1/3, linewidth=0)
            ax.fill_between(r_ref, y, 0, where=(y >= 0), color='r', alpha=1/3, linewidth=0)
            ax.set_ylabel(ylabel)
            ax.set_ylim(ymin, ymax)

        for row in range(4):
            for col in range(2):
                ax = axs[row, col]
                ax.set_xscale('log')
                ax.set_xticks([2, 4, 6, 8, 10, 20, 30])
                ax.xaxis.set_major_formatter(ScalarFormatter())
                ax.minorticks_off()
                ax.xaxis.set_minor_formatter(plt.NullFormatter())
                if row == 3:
                    ax.set_xlabel(r'$r$')

        axs[0, 0].set_title('Disk')
        axs[0, 1].set_title('Corona')
        for row in range(4):
            axs[row, 1].tick_params(labelleft=False)

        if 'STAR' in path:
            axs[0, 1].plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            axs[0, 1].plot([], [], ' ', label=f't = {times[i]:.0f}')
        axs[0, 1].legend(loc='upper right')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    gif_path = f'{path}/Storage/panel.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))

### 1. Plots

In [3]:
path = 'BH_VISC_HD/'
animate_density(path)
animate_velocity(path)
animate_panel(path)

In [4]:
path = 'BH_VISC_MHD/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

In [5]:
path = 'BH_VISC_RES_MHD/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

In [32]:
path = 'BH_VISC_HD/'
animate_tracer(path)

In [33]:
path = 'BH_VISC_MHD/'
animate_tracer(path)

In [34]:
path = 'BH_VISC_RES_MHD/'
animate_tracer(path)

In [20]:
path = 'BH_VISC_HD_TRC/'
#animate_density(path)
animate_tracer(path, mode='recover')
animate_tracer(path, mode='truth')
#animate_tracer(path, compare=True, panel=False)
#animate_tracer(path, compare=True, panel=True)

In [41]:
path = 'BH_VISC_RES_MHD_TRC/'
animate_density(path)
animate_tracer(path, mode='truth')
animate_tracer(path, mode='recover')
animate_tracer(path, compare=True, panel=False)
animate_tracer(path, compare=True, panel=True)